# T3 — Precision sweep + per-band map (HP-18)

Colab **T4** runner for [[HP-18_Precision_Sweep]] / [[D12_Precision_Compression_Invariance]].

Extras: `bitsandbytes`, `optimum`; `auto-gptq` / `autoawq` when installable. Published GPTQ/AWQ checkpoints only — do not quantize yourself in this HP.

**Secrets:** optional `HF_TOKEN`; `RVC_PUSH_TOKEN` for push.

**Outputs**
- `results/raw/T3_P1_{model_slug}_{precision}.csv`
- `results/derived/T3_precision_metrics.csv`, `T3_band_map.csv`
- `results/figures/T3_band_map.pdf`
- `docs/trackT/T3_REPORT.md`


In [ ]:
# Cell 0 — environment fingerprint (run first; required by HP-16 validate header)
import platform
import subprocess
import sys

print("python:", sys.version.replace("\n", " "))
print("platform:", platform.platform())

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu_name:", torch.cuda.get_device_name(0))
        print("cuda:", torch.version.cuda)
    else:
        print("gpu_name:", None)
        print("cuda:", None)
except Exception as e:
    print("torch: UNAVAILABLE", e)

try:
    import transformers
    print("transformers:", transformers.__version__)
except Exception as e:
    print("transformers: UNAVAILABLE", e)

try:
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], text=True)
    print("nvidia-smi:", out.strip())
except Exception as e:
    print("nvidia-smi: UNAVAILABLE", e)


In [ ]:
# Clone repo (public) or with Colab secret GITHUB_TOKEN for private fetch.
# Never hard-code tokens.
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get(
    "RVC_REPO_URL",
    "https://github.com/Adya6714/retrieval-vs-computation.git",
)
REPO_COMMIT = os.environ.get("RVC_REPO_COMMIT", "")  # empty = default branch HEAD
BRANCH = os.environ.get("RVC_BRANCH", "main")

def _secret(name: str) -> str:
    v = os.environ.get(name, "")
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(name) or ""
    except Exception:
        return ""

GH_TOKEN = _secret("GITHUB_TOKEN") or _secret("RVC_PUSH_TOKEN")
HF_TOKEN = _secret("HF_TOKEN") or _secret("HUGGING_FACE_HUB_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login as _hf_login
        _hf_login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as exc:
        print("[setup] HF login skipped:", exc)

def _looks_like_repo(p: Path) -> bool:
    return (p / "data" / "problems" / "question_bank_gsm.csv").is_file() and (
        p / "scripts" / "trackT"
    ).is_dir()

def _find_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if _looks_like_repo(cand):
            return cand
    return Path("/content/retrieval-vs-computation")

REPO_ROOT = _find_repo()
if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    url = REPO_URL
    if GH_TOKEN and "github.com" in url and url.startswith("https://"):
        url = url.replace("https://", f"https://{GH_TOKEN}@")
    print(f"[setup] cloning {REPO_URL} → {REPO_ROOT}")
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", BRANCH, url, str(REPO_ROOT)])
    if REPO_COMMIT:
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", REPO_COMMIT])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", REPO_COMMIT])

assert _looks_like_repo(REPO_ROOT), f"Repo not found at {REPO_ROOT}"
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"[setup] REPO_ROOT={REPO_ROOT}")
print(f"[setup] HEAD=", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
# Install repo requirements + Track T extras (restart runtime if bitsandbytes was just added).
import subprocess
import sys
from pathlib import Path

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
req = Path("requirements.txt")
assert req.is_file(), "run the clone cell first"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

extras = [
    "torch",
    "transformers>=4.44",
    "accelerate>=0.33",
    "huggingface_hub",
    "sentencepiece",
    "protobuf",
    "bitsandbytes>=0.43",
    "optimum>=1.21",
]
# GPTQ / AWQ loaders — optional; HP-18 skips arms when published checkpoints are missing.
for pkg in ["auto-gptq", "autoawq"]:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", pkg])
        print(f"[pip] {pkg} ok")
    except subprocess.CalledProcessError as e:
        print(f"[pip] {pkg} skipped ({e})")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *extras])
print("[pip] T3 extras installed")


## Run HP-18 precision + band arms with `--resume`


In [ ]:
DRY_RUN = False
MODELS = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "Qwen/Qwen2.5-Math-1.5B-Instruct",
]
# fp16 reuses T2 when present; int8/nf4 via bitsandbytes; gptq/awq skip if unavailable.
PRECISIONS = ["fp16", "int8", "nf4", "gptq_int4", "awq_int4"]

from pathlib import Path
import subprocess, sys

assert Path("scripts/trackT/T3_precision_sweep.py").is_file()
assert Path("scripts/trackT/fake_quant.py").is_file()

for model in MODELS:
    for prec in PRECISIONS:
        cmd = [
            sys.executable, "scripts/trackT/T3_precision_sweep.py",
            "--model", model,
            "--precision", prec,
            "--resume",
        ]
        if DRY_RUN:
            cmd.append("--dry-run")
        print(">>", " ".join(cmd))
        subprocess.check_call(cmd)

    # Per-band fake INT4 (4 contiguous decoder bands)
    for band in range(4):
        cmd = [
            sys.executable, "scripts/trackT/fake_quant.py",
            "--model", model,
            "--band", str(band),
            "--n-bands", "4",
            "--resume",
        ]
        if DRY_RUN:
            cmd.append("--dry-run")
        print(">>", " ".join(cmd))
        subprocess.check_call(cmd)

print(">> metrics")
subprocess.check_call([sys.executable, "scripts/trackT/T3_precision_metrics.py"])


## Push raw + derived (requires `RVC_PUSH_TOKEN`)


In [ ]:
# Push raw/derived Track T artefacts. Requires Colab secret RVC_PUSH_TOKEN
# (or GITHUB_TOKEN) with contents:write. Never hard-code a token.
from __future__ import annotations

import os
import subprocess
from pathlib import Path

def _secret(name: str) -> str:
    v = os.environ.get(name, "")
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(name) or ""
    except Exception:
        return ""

TOKEN = _secret("RVC_PUSH_TOKEN") or _secret("GITHUB_TOKEN")
assert TOKEN, "Set Colab secret RVC_PUSH_TOKEN (preferred) or GITHUB_TOKEN before pushing."

root = Path.cwd()
assert (root / "results" / "raw").is_dir()

subprocess.check_call(["git", "config", "user.email", "colab-trackt@users.noreply.github.com"])
subprocess.check_call(["git", "config", "user.name", "RvC Track T Colab"])
subprocess.check_call(["git", "add", "results/raw", "results/derived", "results/figures", "docs/trackT"])
status = subprocess.check_output(["git", "status", "--porcelain"], text=True)
if not status.strip():
    print("[push] nothing to commit")
else:
    print(status)
    msg = os.environ.get("RVC_COMMIT_MSG", "Track T Colab: append raw/derived outputs")
    subprocess.check_call(["git", "commit", "-m", msg])
    # rewrite origin to authenticated URL for this push only
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], text=True).strip()
    if origin.startswith("https://") and "@" not in origin.split("://", 1)[1]:
        auth = origin.replace("https://", f"https://x-access-token:{TOKEN}@")
        subprocess.check_call(["git", "remote", "set-url", "origin", auth])
    branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip()
    subprocess.check_call(["git", "push", "-u", "origin", "HEAD"])
    # scrub token from remote URL
    if "x-access-token:" in origin or origin.startswith("https://"):
        clean = origin
        if "x-access-token:" in clean:
            # already cleaned below
            pass
        clean = subprocess.check_output(["git", "remote", "get-url", "origin"], text=True).strip()
        if "x-access-token:" in clean:
            scrubbed = "https://github.com/" + clean.split("github.com/")[-1]
            subprocess.check_call(["git", "remote", "set-url", "origin", scrubbed])
    print("[push] done on", branch)
